# Kerberos Protocol — vLLM Runner (full battery)

Run a HuggingFace model through **all** Kerberos interrogation techniques on a local **vLLM** OpenAI-compatible server, score every session, then download a single zip with the results.

**Default model:** `paperscarecrow/Gemma-4-31B-it-abliterated` (Q8_0 GGUF).
At Q8, ~33 GB of weights — needs an **A100 40GB+ / H100 / H200**. Won't fit on Colab Free (T4) or L4.

**Pipeline:**
1. Clone the repo
2. Install deps (vllm, openai, anthropic, weasyprint, python-dotenv, huggingface_hub)
3. Load API keys (Colab Secrets → `.env` → existing env → inline)
4. Download the GGUF and launch vLLM on `localhost:8000`
5. Run every technique in order, chaining findings forward (WAT → shadow → narrative → AI → stems)
6. Score each session with the LLM rater
7. Zip everything and download the bundle

**Which keys you need:**
- `OPENROUTER_API_KEY` *or* `ANTHROPIC_API_KEY` for the **interrogator**. If neither is set, the local Gemma plays both roles (weak but free).
- `ANTHROPIC_API_KEY` preferred for the **rater** (scoring).
- `HF_TOKEN` if the GGUF repo / tokenizer repo is gated.

## 1. Clone the repo

In [ ]:
import os, subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/ChuloIva/RL_rh"
REPO_DIR = Path.cwd() / "RL_rh"

# If this notebook lives inside an already-cloned copy, use that instead
if (Path.cwd() / "deep" / "runner.py").exists():
    REPO_DIR = Path.cwd()
    print(f"Using existing repo at {REPO_DIR}")
elif REPO_DIR.exists():
    print(f"Repo already cloned at {REPO_DIR}")
else:
    subprocess.check_call(["git", "clone", REPO_URL, str(REPO_DIR)])
    print(f"Cloned into {REPO_DIR}")

os.chdir(REPO_DIR)
print("cwd:", os.getcwd())

## 2. Install dependencies

vLLM is the heavy one — first install takes a few minutes.

In [ ]:
# We use llama-cpp-python's server instead of vLLM. Reasons:
#   • vLLM ≥ 0.20 is the only version that knows the Gemma-4 architecture, but it
#     ships against CUDA 13, which Colab runtimes don't have, and the cu13 backfill
#     packages don't have py3.12 wheels yet.
#   • llama.cpp loads GGUF natively, works with whatever CUDA is on the box, and
#     llama_cpp.server exposes the same OpenAI-compatible /v1/chat/completions API.
# So the rest of the notebook (OpenAI SDK calls) keeps working unchanged.

%pip install -q openai anthropic weasyprint python-dotenv huggingface_hub
%pip install -q "llama-cpp-python[server]" \
    --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124
# If Colab is on a different CUDA, swap the index URL — abetlen publishes wheels for
# cu121, cu122, cu123, cu124, cu125. Check `nvcc --version` if unsure.

## 3. API keys

`.env` is gitignored, so cloning the repo does **not** bring your keys with it. Pick one of these:

**Colab Secrets (recommended on Colab):** click the 🔑 icon in the left sidebar → *Add new secret* → name it `ANTHROPIC_API_KEY` (and/or `OPENROUTER_API_KEY`, `HF_TOKEN`) → paste the value → toggle **Notebook access** on. The cell below pulls them into env vars.

**Local `.env` file:** put `ANTHROPIC_API_KEY=sk-ant-...` (one per line) in `RL_rh/.env`. Loaded automatically below.

**Inline (quick & dirty):** uncomment the `os.environ[...]` lines below and paste. **Do not commit** the notebook after doing this.

In [ ]:
# ---- Inline fallback: uncomment and paste, then re-run this cell ----
# os.environ["ANTHROPIC_API_KEY"]  = "sk-ant-..."
# os.environ["OPENROUTER_API_KEY"] = "sk-or-..."
# os.environ["HF_TOKEN"]           = "hf_..."

_KEYS = ["ANTHROPIC_API_KEY", "OPENROUTER_API_KEY", "OPENAI_API_KEY", "HF_TOKEN"]

# 1) Colab Secrets — only available inside Colab with notebook-access enabled
try:
    from google.colab import userdata  # type: ignore
    for _k in _KEYS:
        if not os.environ.get(_k):
            try:
                _v = userdata.get(_k)
                if _v:
                    os.environ[_k] = _v
            except Exception:
                pass   # secret not defined or access not granted — skip silently
    print("Checked Colab Secrets.")
except ImportError:
    pass   # not on Colab

# 2) .env file — looks in the repo root and the current dir
try:
    from dotenv import load_dotenv
    for _p in [REPO_DIR / ".env", Path.cwd() / ".env"]:
        if _p.exists():
            load_dotenv(_p, override=False)
            print(f"Loaded {_p}")
except ImportError:
    pass

# Report status (never print the keys themselves)
for _k in _KEYS:
    print(f"  {_k:20s}: {'set' if os.environ.get(_k) else '—'}")

if not (os.environ.get("ANTHROPIC_API_KEY") or os.environ.get("OPENROUTER_API_KEY")):
    print("\n[warn] No interrogator/rater API key found. The notebook will fall back to using the")
    print("       local vLLM model for ALL roles (self-interrogation + self-rating). Cheap, but weak.")

## 4. Configuration

Edit these to taste. If the GGUF filename below doesn't match what's actually in the repo, the launch cell will fail — check the file listing on https://huggingface.co/paperscarecrow/Gemma-4-31B-it-abliterated and adjust.

In [ ]:
# === Target model (the one being tested) ===
# Gemma-4 31B abliterated, Q8_0 GGUF.
HF_MODEL = "paperscarecrow/Gemma-4-31B-it-abliterated"

# Exact GGUF filename in the repo. Other available quants:
#   gemma-4-31b-abliterated-Q4_K_M.gguf (18.7 GB)  — fits on a single 24 GB GPU
#   gemma-4-31b-abliterated-Q8_0.gguf  (32.6 GB)  — needs A100 40GB+ / H100
#   gemma-4-31b-abliterated-f16.gguf   (61.4 GB)  — needs H100 80GB+ / 2× A100
HF_GGUF_FILE = "gemma-4-31b-abliterated-Q8_0.gguf"

# === Server settings ===
SERVER_PORT = 8000
SERVER_HOST = "127.0.0.1"
SERVER_BASE_URL = f"http://{SERVER_HOST}:{SERVER_PORT}/v1"
N_CTX = 8192               # context window for llama-server
N_GPU_LAYERS = -1          # -1 = offload all layers; reduce if you OOM

# (Legacy alias kept so other cells that still reference VLLM_* don't break.)
VLLM_PORT, VLLM_HOST, VLLM_BASE_URL = SERVER_PORT, SERVER_HOST, SERVER_BASE_URL

# === Techniques to run, in order (findings chain forward into the next) ===
TECHNIQUES = [
    "word_association_test",
    "shadow_probing",
    "narrative_elicitation",
    "active_imagination",
    "loevinger_stems",
]
MAX_TURNS = 20   # per technique. The original Llama/Gemini runs used 40–70.

# === Interrogator (the one asking the questions) ===
if os.environ.get("OPENROUTER_API_KEY"):
    INTERROGATOR_SPEC = "openrouter:anthropic/claude-sonnet-4"
elif os.environ.get("ANTHROPIC_API_KEY"):
    INTERROGATOR_SPEC = "anthropic:claude-sonnet-4-6"
else:
    INTERROGATOR_SPEC = f"openai:{HF_MODEL}"   # self-interrogation via the local server
    print("No OPENROUTER/ANTHROPIC key — using the local model as interrogator too.")

# === Scorer rater ===
if os.environ.get("ANTHROPIC_API_KEY"):
    RATER_SPEC = "anthropic:claude-opus-4-7"
elif os.environ.get("OPENROUTER_API_KEY"):
    RATER_SPEC = "openrouter:anthropic/claude-opus-4"
else:
    RATER_SPEC = f"openai:{HF_MODEL}"

print("Target       :", f"openai:{HF_MODEL}", "(via local llama-server)")
print("  GGUF file  :", HF_GGUF_FILE)
print("Interrogator :", INTERROGATOR_SPEC)
print("Rater        :", RATER_SPEC)
print("Techniques   :", TECHNIQUES, f"({MAX_TURNS} turns each)")

## 5. Download the GGUF and launch llama-server

Downloads the single `.gguf` file (not the whole repo), then spawns `llama_cpp.server` as a subprocess and polls `/v1/models` until it's ready. The model is served under the name `HF_MODEL` (via `--model_alias`) so all later code can reference it that way regardless of where the file lives on disk. llama.cpp reads the tokenizer + chat template directly from the GGUF, so no separate tokenizer download is needed.

In [ ]:
import time, signal, atexit, urllib.request

SERVER_LOG = Path("/tmp/llama_server.log")
VLLM_LOG = SERVER_LOG   # back-compat alias for the zip cell

def server_alive():
    try:
        with urllib.request.urlopen(f"{SERVER_BASE_URL}/models", timeout=2) as r:
            return r.status == 200
    except Exception:
        return False

server_proc = None
vllm_proc = None   # back-compat alias for the cleanup cell
if server_alive():
    print(f"llama-server already running on port {SERVER_PORT} — reusing.")
else:
    from huggingface_hub import hf_hub_download

    print(f"Downloading {HF_GGUF_FILE} from {HF_MODEL} (large — be patient)…")
    gguf_path = hf_hub_download(
        repo_id=HF_MODEL,
        filename=HF_GGUF_FILE,
        token=os.environ.get("HF_TOKEN"),
    )
    print(f"  → {gguf_path}")

    cmd = [
        sys.executable, "-m", "llama_cpp.server",
        "--model", gguf_path,
        "--host", SERVER_HOST,
        "--port", str(SERVER_PORT),
        "--n_gpu_layers", str(N_GPU_LAYERS),   # -1 = offload everything
        "--n_ctx", str(N_CTX),
        "--model_alias", HF_MODEL,             # makes /v1/models report HF_MODEL
    ]
    print("Launching:", " ".join(cmd))
    print("Log:", SERVER_LOG)
    log_f = open(SERVER_LOG, "w")
    server_proc = subprocess.Popen(cmd, stdout=log_f, stderr=subprocess.STDOUT)
    vllm_proc = server_proc

    def _cleanup():
        if server_proc and server_proc.poll() is None:
            print("Stopping llama-server…")
            server_proc.send_signal(signal.SIGINT)
            try: server_proc.wait(timeout=15)
            except subprocess.TimeoutExpired: server_proc.kill()
    atexit.register(_cleanup)

    # poll until alive (or process dies). 31B GGUF takes a few minutes to mmap + warm up.
    deadline = time.time() + 1800
    while time.time() < deadline:
        if server_proc.poll() is not None:
            raise RuntimeError(f"llama-server exited early. See {SERVER_LOG}")
        if server_alive():
            break
        time.sleep(3)
    else:
        raise TimeoutError(f"llama-server never came up. See {SERVER_LOG}")

print("llama-server is ready:", SERVER_BASE_URL)

## 6. Point the OpenAI SDK at the local server

`runner.create_client("openai")` calls `openai.OpenAI()` with no args, which reads `OPENAI_BASE_URL` and `OPENAI_API_KEY` from the env. We override both to route to llama-server. (Anthropic and OpenRouter clients build their own URLs, so they ignore these overrides — your real keys still work for them.)

In [ ]:
os.environ["OPENAI_BASE_URL"] = VLLM_BASE_URL
os.environ["OPENAI_API_KEY"] = "vllm-local"   # vLLM doesn't validate it

# Sanity-check end-to-end
from openai import OpenAI
_test = OpenAI().chat.completions.create(
    model=HF_MODEL,
    messages=[{"role": "user", "content": "Reply with exactly: pong"}],
    max_tokens=8,
)
print("vLLM says:", _test.choices[0].message.content)

## 7. Run all techniques

Loops over `TECHNIQUES`, running each one and feeding the previous run's findings into the next. Outputs land in `deep/sessions/`.

In [ ]:
sys.path.insert(0, str(REPO_DIR / "deep"))
from runner import run_session

sessions_dir = REPO_DIR / "deep" / "sessions"
session_paths = []
findings_path = None   # chained forward between techniques
run_errors = {}

for tech in TECHNIQUES:
    print(f"\n{'='*70}\n RUN: {tech}\n{'='*70}")
    technique_path = REPO_DIR / "deep" / "techniques" / f"{tech}.json"
    if not technique_path.exists():
        print(f"  [skip] no such technique file: {technique_path}")
        continue
    try:
        run_session(
            technique_path=str(technique_path),
            interrogator_spec=INTERROGATOR_SPEC,
            target_spec=f"openai:{HF_MODEL}",
            max_turns=MAX_TURNS,
            findings_path=findings_path,
            auto_extract=True,
        )
    except Exception as e:
        print(f"  [error] {tech}: {e}")
        run_errors[tech] = str(e)
        continue

    # locate the session file we just wrote (newest match for this technique)
    matches = sorted(sessions_dir.glob(f"*{tech}*.json"), key=lambda p: p.stat().st_mtime)
    matches = [p for p in matches if not p.name.endswith("_findings.json") and not p.name.endswith("_scores.json")]
    if not matches:
        print(f"  [warn] couldn't find session file for {tech}")
        continue
    session_path = matches[-1]
    session_paths.append(session_path)

    # chain: feed this run's findings into the next technique
    fp = session_path.with_name(session_path.stem + "_findings.json")
    if fp.exists():
        findings_path = str(fp)
        print(f"  → chaining findings: {fp.name}")

print(f"\nAll done — {len(session_paths)}/{len(TECHNIQUES)} techniques completed.")
if run_errors:
    print("Errors:", run_errors)

## 8. Score every session

Runs the scorer pipeline against each completed session.

In [ ]:
import os, sys, json, importlib

# 1) If session_paths got lost (kernel restart, re-run, etc.), rebuild from disk.
sessions_dir = REPO_DIR / "deep" / "sessions"
if 'session_paths' not in dir() or not session_paths:
    safe_model = HF_MODEL.replace("/", "_").replace(":", "_")
    session_paths = sorted(
        p for p in sessions_dir.glob(f"{safe_model}_*.json")
        if not p.name.endswith("_findings.json") and not p.name.endswith("_scores.json")
    )
    print(f"Recovered {len(session_paths)} sessions from disk")

# 2) Pick rater — we have OpenRouter, not Anthropic.
print(f"ANTHROPIC_API_KEY in env : {'yes' if os.environ.get('ANTHROPIC_API_KEY') else 'no'}")
print(f"OPENROUTER_API_KEY in env: {'yes' if os.environ.get('OPENROUTER_API_KEY') else 'no'}")

if os.environ.get("OPENROUTER_API_KEY"):
    RATER_SPEC = "openrouter:anthropic/claude-sonnet-4.6"
elif os.environ.get("ANTHROPIC_API_KEY"):
    RATER_SPEC = "anthropic:claude-opus-4-7"
else:
    raise RuntimeError("Set ANTHROPIC_API_KEY or OPENROUTER_API_KEY before running.")
print(f"Rater: {RATER_SPEC}")

# 3) Repopulate INSTRUMENT_REGISTRY. The earlier reload of scorers.base created
#    a fresh empty registry dict; the scorer modules already ran their @register
#    calls against the *old* dict (now orphaned). Evict them from sys.modules
#    and re-import so the decorators fire against the current registry.
sys.path.insert(0, str(REPO_DIR / "deep"))
to_evict = [m for m in list(sys.modules)
            if m.startswith("scorers.") and m not in ("scorers.base", "scorers")]
for m in to_evict:
    del sys.modules[m]
print(f"Evicted {len(to_evict)} scorer modules from sys.modules")

# Make sure scorers.base itself is healthy (re-import the current empty registry)
import scorers.base
importlib.reload(scorers.base)
scorers.base._ensure_scorers_imported()
from scorers.base import INSTRUMENT_REGISTRY
print(f"Scorers registered: {len(INSTRUMENT_REGISTRY)} → {sorted(INSTRUMENT_REGISTRY.keys())}")

# 4) Patch each scorer module's DEFAULT_RATER (their `from _rater import DEFAULT_RATER`
#    created bound copies). Now that they've been re-imported, do this on the
#    freshly-loaded ones.
patched = []
for mod_name, mod in list(sys.modules.items()):
    if mod_name.startswith("scorers.") and hasattr(mod, "DEFAULT_RATER"):
        setattr(mod, "DEFAULT_RATER", RATER_SPEC)
        patched.append(mod_name)
print(f"Patched DEFAULT_RATER in {len(patched)} modules")

# 5) Now reload score_session so its filter_scorers() sees the rebuilt registry.
if "score_session" in sys.modules:
    importlib.reload(sys.modules["score_session"])
from score_session import score_session

# 6) Score every recovered session.
all_scores = {}
score_errors = {}
for session_path in session_paths:
    print(f"\n--- Scoring: {session_path.name} ---")
    try:
        scores = score_session(session_path, rater=RATER_SPEC)
        all_scores[session_path.stem] = scores
        for instrument, result in scores.get("results", {}).items():
            s = result.get("scores", {})
            print(f"  {instrument}: {json.dumps(s, ensure_ascii=False)[:140]}")
    except Exception as e:
        print(f"  [error] {e}")
        score_errors[session_path.stem] = str(e)

print(f"\nScored {len(all_scores)}/{len(session_paths)} sessions.")
if score_errors:
    print("Errors:", score_errors)

## 9. Bundle results into a zip and download

Packages every transcript / log / findings / scores file produced in this run, plus a summary and the vLLM log, into a single zip. On Colab it triggers a browser download; locally it just leaves the zip in the repo root.

In [ ]:
import zipfile, json
from datetime import datetime, timezone

safe_model = HF_MODEL.replace("/", "_").replace(":", "_")
timestamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
zip_path = REPO_DIR / f"kerberos_results_{safe_model}_{timestamp}.zip"

summary = {
    "model": HF_MODEL,
    "gguf_file": HF_GGUF_FILE,
    "interrogator": INTERROGATOR_SPEC,
    "rater": RATER_SPEC,
    "techniques": TECHNIQUES,
    "max_turns": MAX_TURNS,
    "timestamp": timestamp,
    "sessions": [p.name for p in session_paths],
    "run_errors": run_errors if 'run_errors' in dir() else {},
    "score_errors": score_errors if 'score_errors' in dir() else {},
    "scores": all_scores,
}

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    # bundle every file produced by every session this run (transcript, log, findings, scores)
    for session_path in session_paths:
        stem = session_path.stem
        for f in (REPO_DIR / "deep" / "sessions").glob(f"{stem}*"):
            zf.write(f, arcname=f"sessions/{f.name}")
    zf.writestr("run_summary.json", json.dumps(summary, indent=2, ensure_ascii=False))
    # also include the llama-server log for debugging
    _log = globals().get("SERVER_LOG") or globals().get("VLLM_LOG")
    if _log and Path(_log).exists():
        zf.write(_log, arcname="server.log")

print(f"Wrote {zip_path} ({zip_path.stat().st_size / 1024:.1f} KB)")

# Trigger browser download on Colab; otherwise just print the path
try:
    from google.colab import files  # type: ignore
    files.download(str(zip_path))
except ImportError:
    print(f"Local run — grab the file from: {zip_path}")

## 10. Cleanup

Run this when you're done to free GPU memory.

In [ ]:
if 'vllm_proc' in dir() and vllm_proc and vllm_proc.poll() is None:
    vllm_proc.send_signal(signal.SIGINT)
    try: vllm_proc.wait(timeout=15)
    except subprocess.TimeoutExpired: vllm_proc.kill()
    print("vLLM stopped.")